# Required libraries

In [ ]:
import numpy as np
import pandas as pd
#import matplotlib.pyplot as plt
#import seaborn as sns
import tensorflow as tf
from tensorflow import keras
#from keras.optimizers import Adam
from keras.losses import Loss
from keras.initializers import HeNormal, HeUniform, GlorotUniform
from keras.utils import to_categorical
from joblib import Parallel, delayed, parallel_backend
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

## GPU visibility

In [ ]:
# To disable GPU usage, uncomment the following lines:
#import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [ ]:
# print(tf.config.experimental.list_physical_devices())
# gpus = tf.config.list_physical_devices('GPU'); print(gpus)
# tf.config.set_visible_devices([gpus[0]], 'GPU')
# tf.config.get_visible_devices('GPU')

In [ ]:
# To prevent GPU memory fragmentation, you can set a memory limit (adjust as needed):
# gpus = tf.config.get_visible_devices('GPU') # tf.config.list_physical_devices('GPU')
# if gpus:
#     try:
#         for gpu in gpus:
#             #tf.config.experimental.set_memory_growth(gpu, True)
#             tf.config.experimental.set_virtual_device_configuration(gpu, [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=3774)])
#             print(f"Enabled memory growth for GPU: {gpu.name}")
#         logical_gpus = tf.config.list_logical_devices('GPU')
#         print(f"{len(gpus)} Physical GPUs, {len(logical_gpus)} Logical GPUs configured.")
#     except RuntimeError as e:
#         print(f"RuntimeError during GPU configuration: {e}")
#         print("Ensure GPU memory configuration is set at the very beginning of your script.")
# else:
#     print("No GPU devices found. Running on CPU.")

# Load your dataset

In [ ]:
# Load your dataset here. For example, if you are using the CIFAR-10 dataset, you can load it as follows:
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train[:,0] # Flatten the labels to a 1D array
y_test = y_test[:,0] # Flatten the labels to a 1D array
# No need for manual one-hot encoding since everything is handled by the loss functions

X_train = X_train / 255.0 # Normalize the pixel values to [0, 1]
X_test = X_test / 255.0 # Normalize the pixel values to [0, 1]

In [ ]:
# Contamination introducer, if you want to artificially corrupt the labels to test robustness
# def corrupt_labels(y_train, prob, num_classes=10, seed=None):
#     if seed is not None:
#         np.random.seed(seed)

#     y_corrupted = y_train.copy()
#     n = len(y_train)
#     # Decide which labels to corrupt
#     corrupt_mask = np.random.rand(n) < prob
#     # For each label to corrupt, choose a new label different from the original
#     for i in np.where(corrupt_mask)[0]:
#         original_label = y_train[i]
#         # possible new labels excluding the original
#         new_labels = list(range(num_classes))
#         new_labels.remove(original_label)
#         # randomly pick a new label
#         y_corrupted[i] = np.random.choice(new_labels)

#     return y_corrupted

# Different robust loss functions, as Python objects

In [ ]:
# SD-loss implementation
class SDIV(Loss):
    def __init__(self, beta, lam, trim_ratio): # beta and lambda are the SD tuning parameter. Set trim_ratio to 0 to disable trimming, as done in our paper.
        super().__init__()
        self.beta = float(beta)
        self.lam = float(lam)
        self.trim_ratio = trim_ratio

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)  # Clip predictions
        
        A = 1 + self.lam*(1 - self.beta)
        B = self.beta - self.lam*(1 - self.beta)
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses =  (tf.reduce_sum(y_pred**(self.beta+1), axis=1))/A - ((1+self.beta)/(A*B))*(sel_probs**B)
        sorted_losses = tf.sort(losses)
        k = tf.cast(tf.math.floor((1.0 - self.trim_ratio) * tf.cast(batch_size, tf.float32)), tf.int32)
        trimmed_losses = sorted_losses[:k]  # Keep only k smallest residuals
        
        return tf.reduce_mean(trimmed_losses)

# TSCCE loss implementation
class TSCCE(Loss):
    def __init__(self, trim_ratio=0.2):
        super().__init__()
        self.trim_ratio = trim_ratio

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        # Clip predictions to avoid log(0)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        log_probs = tf.math.log(y_pred)
        # Get log probability of the correct class for each sample
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        per_sample_loss = -tf.gather_nd(log_probs, indices)
        # Trim top X% highest-loss samples
        k = tf.cast(tf.math.floor((1.0 - self.trim_ratio) * tf.cast(batch_size, tf.float32)), tf.int32)
        trimmed_values, _ = tf.math.top_k(-per_sample_loss, k=k, sorted=False)
        trimmed_loss = -tf.reduce_mean(trimmed_values)

        return trimmed_loss

# DPD loss implementation
class TDPDSCCE(Loss):
    def __init__(self, beta, trim_ratio): # beta is the DPD tuning parameter. set trim_ratio to 0 to disable trimming, as done in our paper
        super().__init__()
        self.beta = float(beta)
        self.trim_ratio = trim_ratio

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)  # Clip predictions
        
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses =  tf.reduce_sum(y_pred**(self.beta+1), axis=1) - (1+1/self.beta)*(sel_probs**self.beta)
        sorted_losses = tf.sort(losses)
        k = tf.cast(tf.math.floor((1.0 - self.trim_ratio) * tf.cast(batch_size, tf.float32)), tf.int32)
        trimmed_losses = sorted_losses[:k]  # Keep only k smallest residuals
        
        return tf.reduce_mean(trimmed_losses)

# SCE loss implementation
class SCE(Loss):
    def __init__(self, alpha, beta):
        super().__init__()
        self.alpha = float(alpha)
        self.beta = float(beta)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0) # Clip predictions
        
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses = -self.alpha*tf.math.log(sel_probs) + self.beta*6*(1 - sel_probs)

        return tf.reduce_mean(losses)

# GCE loss implementation
class GCE(Loss):
    def __init__(self, q):
        super().__init__()
        self.q = float(q)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        # Clip predictions
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)
        
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses =  (1 - (sel_probs**self.q))/self.q

        return tf.reduce_mean(losses)

# RKLD loss implementation
class RKLD(Loss):
    def __init__(self):
        super().__init__()

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)  # Clip predictions
        
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        
        losses =  tf.reduce_sum(y_pred*tf.math.log(y_pred), axis=1) + 2*(1 - sel_probs)        
        return tf.reduce_mean(losses)

# FCL loss implementation
class FCL(Loss):
    def __init__(self, mu):
        super().__init__()
        self.mu = float(mu)

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32) # Clip predictions
        y_pred = tf.clip_by_value(y_pred, 1e-9, 1.0)
        
        batch_size = tf.shape(y_true)[0]
        indices = tf.stack([tf.range(batch_size), y_true], axis=1)
        sel_probs = tf.gather_nd(y_pred, indices)
        losses =   (-tf.math.log(sel_probs))**(1-self.mu)/tf.exp(tf.math.lgamma(tf.constant(2-self.mu))) + 2*(1-sel_probs)

        return tf.reduce_mean(losses)

# Model

In [ ]:
# Define a CNN model architecture
# Similarly define your model architecture here. You can change the number of layers, filters, and other hyperparameters as needed.
def get_model():
    model = keras.Sequential([
        keras.layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu', kernel_initializer=HeNormal(seed=42)),
        keras.layers.MaxPooling2D((2, 2)),
        
        keras.layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu', kernel_initializer=HeNormal(seed=42)),
        keras.layers.MaxPooling2D((2, 2)),
        
        keras.layers.Flatten(),
        keras.layers.Dense(512, activation='relu', kernel_initializer=HeNormal(seed=42)),
        keras.layers.Dense(10, activation='softmax', kernel_initializer=GlorotUniform(seed=42))     
    ])
    return model

# rSDNet $(\beta, \lambda)$

In [ ]:
n_epochs = 250 # You can change the number of epochs here.
tf.random.set_seed(42)
model = get_model()
model.compile(optimizer='adam', loss=SDIV(beta=0.05, lam=-0.8, trim_ratio=0.0)) # You can chang beta and lambda here. Set trim_ratio to 0 to get the untrimmed version of the SDIV loss, as done in our paper
model.fit(X_train, y_train, epochs = n_epochs, verbose=0)
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy

# CCE

In [ ]:
n_epochs = 250 # You can change the number of epochs here. 
np.random.seed(42)
tf.random.set_seed(42)
model = get_model()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.fit(X_train, y_train, epochs = n_epochs, verbose=0)
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy

# TSCCE

In [ ]:
n_epochs = 250 # You can change the number of epochs for training
np.random.seed(42)
tf.random.set_seed(42)
model = get_model()
model.compile(optimizer='adam', loss=TSCCE(trim_ratio=0.2)) # You can change the trim_ratio
model.fit(X_train, y_train, epochs = n_epochs, verbose=0)
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy

# GCE

In [ ]:
n_epochs = 250 # You can change the number of epochs
np.random.seed(42)
tf.random.set_seed(42)
model = get_model()
model.compile(optimizer='adam', loss=GCE(q=0.7)) # You can change the q value
model.fit(X_train, y_train, epochs = n_epochs, verbose=0)
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy

# SCE

In [ ]:
n_epochs = 250 # You can change the number of epochs
np.random.seed(42)
tf.random.set_seed(42)
model = get_model()
model.compile(optimizer='adam', loss=SCE(alpha=0.5, beta=1.0)) # You can change the alpha and beta values
model.fit(X_train, y_train, epochs = n_epochs, verbose=0)
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy

# FCL

In [ ]:
n_epochs = 250 # You can change the number of epochs
np.random.seed(42)
tf.random.set_seed(42)
model = get_model()
model.compile(optimizer='adam', loss=FCL(mu=0.5)) # You can change the mu value
model.fit(X_train, y_train, epochs = n_epochs, verbose=0)
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy

# TDPD

In [ ]:
n_epochs = 250 # You can change the number of epochs as needed
np.random.seed(42)
tf.random.set_seed(42)
model = get_model()
model.compile(optimizer='adam', loss=TDPDSCCE(beta=0.5, trim_ratio=0)) # You can change the beta here. Set trim_ratio to 0 to get the untrimmed version of the DPD loss, as done in our paper
model.fit(X_train, y_train, epochs = n_epochs, verbose=0)
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy

# MAE

In [ ]:
n_epochs = 250 # You can change the number of epochs as needed
np.random.seed(42)
tf.random.set_seed(42)
model = get_model()
y_train_corrupted = to_categorical(y_train, 10)
model.compile(optimizer='adam', loss='mae')
model.fit(X_train, y_train_corrupted, epochs = n_epochs, verbose=0)
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy

# RKLD

In [ ]:
n_epochs = 250 # You can change the number of epochs as needed
np.random.seed(42)
tf.random.set_seed(42)
model = get_model()
model.compile(optimizer='adam', loss=RKLD())
model.fit(X_train, y_train_corrupted, epochs = 250, verbose=0)
y_pred = model.predict(X_test, verbose=0)
y_pred_labels = [np.argmax(i) for i in y_pred]

print('Accuracy:', accuracy_score(y_test, y_pred_labels)) # Gives the test data accuracy